<a href="https://colab.research.google.com/github/ayoushbirjani4-spec/fiyrank-firstlab/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayoushbirjani4-spec/fiyrank-firstlab/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [20]:
%pip -q install duckdb huggingface_hub

import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN so the prompt never fires (repo is public — never paste the token in a cell).
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# COUNT(*) over Parquet touches metadata, not data — near-free, confirms we're pointed at the right thing.
for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:16} {n:>12,} rows')

dim_clients               104 rows
dim_content           519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily         78,835,655 rows
fact_query_90d      2,414,248 rows


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Lane: Refresh / Content Opportunity Scoring** — which pages should be reviewed first for
refresh, expansion, or protection.

**One row = one content item (`content_hash_id`), for one client (`client_hash_id`), on one
calendar day (`report_date`)** — that's `fact_content_daily_performance`'s native grain. My
feature frame rolls this up to **one row per content item**, aggregated over a fixed 30-day
decision window, because the refresh decision is made per page, not per page-day.

**Table(s) used:** `fact_content_daily_performance` (time-series signal), joined to `dim_content`
(content metadata, joins) and `dim_clients` (history-coverage checks — `gsc_data_start`,
`ga4_data_start`).

**Time window:** I develop on a **mid-panel month, `month=2026-03`**, not the final month — the
last month of the panel is the natural outcome window for any past→future label, so iterating
there would mean developing inside my own test window. Decision point = start of the window;
features come from the 30 days *before* it, label (in the later modeling weeks) will come from
the 30 days *after* it.

**Predict/rank:** a proxy score for "does this page need review" — for this contract I use the
observable proxy `imp_prev30 < prior period's impressions * 0.8` (a >20% impression drop),
matching the pattern notebook 03 already validated, rather than reusing any FlyRank product flag
(`health_score`, `priority_score` etc. are deliberately **not** in this dataset).

**Deliberately excluded:** any metric measured *after* the decision point (the label window
itself), and the query-level table's raw content beyond aggregate counts — `fact_content_query_90d`
exists to describe query mix, not to be joined row-for-row onto the daily grain (its context
columns repeat per row; `ANY_VALUE()`, never `SUM()`, or the grain check below would catch it).


In [ ]:
# Grain sanity note — the real verification query for this claim lives in section 3, query 1.
# This cell intentionally left as a pointer, not a duplicate query (one grain check, one place).



Secrets
Configure your code by storing environment variables, file paths or keys. Values stored here are private, visible only to you and the notebooks that you select.

Secret name cannot contain spaces.

Notebook access	Name	Value	Actions

HF_TOKEN
•••••••••••••••••••••••••••••••••••••
Access your secret keys in Python via:

from google.colab import userdata
userdata.get('secretName')
ML-04 — Search Intelligence Data Contract
Open In Colab

This skeleton is yours to fill. Work the sections in order — each one has a one-line hint. Simple words, honest numbers.

Working with an AI assistant? Tell it to read skills/README.md first and load the one skill this assignment names on its card.


[20]
15s
%pip -q install duckdb huggingface_hub

import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN so the prompt never fires (repo is public — never paste the token in a cell).
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# COUNT(*) over Parquet touches metadata, not data — near-free, confirms we're pointed at the right thing.
for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:16} {n:>12,} rows')

1. Unit of analysis + time window
One row = one what, over which dates? State it, then verify it below.

Lane: Refresh / Content Opportunity Scoring — which pages should be reviewed first for refresh, expansion, or protection.

One row = one content item (content_hash_id), for one client (client_hash_id), on one calendar day (report_date) — that's fact_content_daily_performance's native grain. My feature frame rolls this up to one row per content item, aggregated over a fixed 30-day decision window, because the refresh decision is made per page, not per page-day.

Table(s) used: fact_content_daily_performance (time-series signal), joined to dim_content (content metadata, joins) and dim_clients (history-coverage checks — gsc_data_start, ga4_data_start).

Time window: I develop on a mid-panel month, month=2026-03, not the final month — the last month of the panel is the natural outcome window for any past→future label, so iterating there would mean developing inside my own test window. Decision point = start of the window; features come from the 30 days before it, label (in the later modeling weeks) will come from the 30 days after it.

Predict/rank: a proxy score for "does this page need review" — for this contract I use the observable proxy imp_prev30 < prior period's impressions * 0.8 (a >20% impression drop), matching the pattern notebook 03 already validated, rather than reusing any FlyRank product flag (health_score, priority_score etc. are deliberately not in this dataset).

Deliberately excluded: any metric measured after the decision point (the label window itself), and the query-level table's raw content beyond aggregate counts — fact_content_query_90d exists to describe query mix, not to be joined row-for-row onto the daily grain (its context columns repeat per row; ANY_VALUE(), never SUM(), or the grain check below would catch it).


[ ]
# Grain sanity note — the real verification query for this claim lives in section 3, query 1.
# This cell intentionally left as a pointer, not a duplicate query (one grain check, one place).

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
2. Fields: feature / label / context / excluded
Sort every field you plan to touch into these four buckets. Excluded needs a why.


[ ]

3. Verify it with queries (grain, counts, missing values, windows)
Every claim above gets a query cell here. A contract claim without a query next to it is a guess.


[ ]
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

4. Data limits
What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.


[ ]
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

Self-check
Before you submit, confirm each line honestly:

 Every section above is filled — markdown thinking AND the code that backs it
 The notebook runs top to bottom with no errors (Runtime → Run all)
 No client names, URLs, or private queries anywhere
 My claims use careful words: observed, measured, directional, decision-support
 Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.
Colab paid products
-
Cancel contracts here


In [ ]:
pass  # bucket table above is the deliverable for this section; queries backing it are in section 3

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1 — Grain check

If one row of the raw daily fact really is one (content, client, day), no
(`content_hash_id`, `client_hash_id`, `report_date`) triple should repeat, for the mid-panel month."

In [21]:
grain_check = con.sql(f"""
    SELECT content_hash_id, client_hash_id, report_date, COUNT(*) AS n
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
    GROUP BY content_hash_id, client_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"Duplicate (content, client, day) triples found: {len(grain_check)}")
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (content, client, day) triples found: 0


,content_hash_id,client_hash_id,report_date,n


### Query 2 — Slice row count + date span

Row count and date span for the `month=2026-03` partition — should match ~30-31 daily rows worth
of content x client combinations, and stay entirely inside March.

In [22]:
slice_stats = con.sql(f"""
    SELECT
        COUNT(*)                    AS n_rows,
        COUNT(DISTINCT content_hash_id) AS n_content_items,
        COUNT(DISTINCT client_hash_id)  AS n_clients,
        MIN(report_date)            AS min_date,
        MAX(report_date)            AS max_date
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
""").df()

slice_stats

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_content_items,n_clients,min_date,max_date
0,9841378,331437,55,2026-03-01,2026-03-31


### Query 3 — Availability, filtered with `IS TRUE`

GA4 sessions are only meaningfully available where `ga4_data_available IS TRUE` (the flag can
also be NULL — a straight `= FALSE` or `NOT ga4_data_available` filter would mishandle those
rows silently, per the schema's documented gotcha). This checks how many March rows actually
survive that filter.

In [23]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)  AS rows_ga4_available,
        COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS rows_ga4_unavailable_or_null
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
""").df()

availability['pct_available'] = availability['rows_ga4_available'] / availability['total_rows']
availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_ga4_available,rows_ga4_unavailable_or_null,pct_available
0,9841378,413966,9427412,0.042064



### Five features (max) — feature frame for `month=2026-03`

Rolled up to one row per content item, using the prior-30-day window inside March relative to a
decision point at day 15 (mid-month, so both a "prior" and "later" half exist inside the same
partition for this contract-verification pass — modeling weeks will use full prior/next-30
windows around a real decision date).

1. **`gsc_impressions_prior`** (summed) — knowable at the decision moment because it's GSC data
   already logged for days strictly before the decision point.
2. **`gsc_clicks_prior`** (summed) — same reasoning; GSC click data through the decision point.
3. **`gsc_avg_position_prior`** (averaged) — same reasoning; position is measured, not predicted.
4. **`ctr_prior`** (derived: `gsc_clicks_prior / NULLIF(gsc_impressions_prior, 0)`) — arithmetic
   on two prior-window columns, so it's knowable exactly when they are.
5. **`content_age_days`** (from `dim_content.content_created_at`, static per item) — knowable
   because it's fixed metadata set when the content was created, always before any decision date.

In [25]:
feature_frame = con.sql(f"""
    WITH prior AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS gsc_impressions_prior,
               SUM(gsc_clicks)      AS gsc_clicks_prior,
               AVG(gsc_avg_position) AS gsc_avg_position_prior
        FROM {TABLES['fact_daily']}
        WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-03-15'
        GROUP BY content_hash_id
    )
    SELECT
        p.content_hash_id,
        p.gsc_impressions_prior,
        p.gsc_clicks_prior,
        p.gsc_avg_position_prior,
        p.gsc_clicks_prior / NULLIF(p.gsc_impressions_prior, 0) AS ctr_prior,
        DATE_DIFF('day', c.content_created_date, DATE '2026-03-15') AS content_age_days
    FROM prior p
    JOIN {TABLES['dim_content']} c USING (content_hash_id)
""").df()

print(f"{len(feature_frame):,} content items in the feature frame")
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

319,585 content items in the feature frame


,content_hash_id,gsc_impressions_prior,gsc_clicks_prior,gsc_avg_position_prior,ctr_prior,content_age_days
0,content_149355c8dfc3f8e1,0.0,0.0,NaN,NaN,52
1,content_14a3d47ccd0d15dc,2.0,0.0,5.5,0.0,96
2,content_14a6f92117604fef,0.0,0.0,NaN,NaN,94
3,content_14a86c63a214f648,0.0,0.0,NaN,NaN,96
4,content_14b1a02c1b8557fb,5.0,0.0,8.0,0.0,94


### The trap — deliberate leakage, then removal

Add one label-derived column on purpose — impressions from the *later* half of the same window,
which wouldn't exist yet at the decision point — fit a quick classifier, watch the score jump
toward 1.0, then delete it and keep the honest number. This is the notebook-02 leakage lesson,
performed here on real warehouse data.

In [26]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

labeled = con.sql(f"""
    WITH prior AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS imp_prior,
               SUM(gsc_clicks)      AS clk_prior,
               AVG(gsc_avg_position) AS pos_prior
        FROM {TABLES['fact_daily']}
        WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-03-15'
        GROUP BY content_hash_id
        HAVING imp_prior >= 50
    ),
    later AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS imp_later
        FROM {TABLES['fact_daily']}
        WHERE report_date >= DATE '2026-03-15' AND report_date < DATE '2026-03-31'
        GROUP BY content_hash_id
    )
    SELECT p.content_hash_id, p.imp_prior, p.clk_prior, p.pos_prior, l.imp_later
    FROM prior p JOIN later l USING (content_hash_id)
""").df().dropna()

# Label: did impressions decline >20% from the prior half to the later half of the same month?
labeled['is_declining'] = (labeled['imp_later'] < 0.8 * labeled['imp_prior']).astype(int)

# LEAK: imp_later is literally the outcome the label is built from — including it as a
# feature isn't a subtle mistake, it's handing the model the answer.
labeled['leaky_feature'] = labeled['imp_later']

honest_features = ['imp_prior', 'clk_prior', 'pos_prior']
leaky_features  = honest_features + ['leaky_feature']

def quick_auc(cols):
    X, y = labeled[cols], labeled['is_declining']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    return roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])

honest_auc = quick_auc(honest_features)
leaky_auc  = quick_auc(leaky_features)

print(f"Honest AUC (prior-window features only): {honest_auc:.3f}")
print(f"Leaky AUC  (+ later-window outcome as a 'feature'): {leaky_auc:.3f}")
print()
print("leaky_feature is deleted below. honest_auc is the number that ships.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Honest AUC (prior-window features only): 0.591
Leaky AUC  (+ later-window outcome as a 'feature'): 1.000

leaky_feature is deleted below. honest_auc is the number that ships.


In [27]:
# Leak removed — this is what actually carries forward.
final_features = honest_features
print("Final honest AUC:", round(honest_auc, 3))

Final honest AUC: 0.591


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation — unbalanced panel, GSC-only early rows.** History depth differs per client
(some clients have 17 months, some 3), and GA4 coverage only starts at each client's own
`ga4_data_start` — rows before that carry `ga4_data_available = FALSE` (or NULL, per query 3
above). Any feature that depends on GA4 sessions is therefore **systematically less available
for newer or newly-tracked clients**, not because those pages have less engagement, but because
they haven't been measured that long. A refresh-priority score that leans on GA4 features will
under-rank young-client pages for a data-coverage reason, not a content-quality reason — that's
a limitation of the slice, not a finding about those pages.

**A second limitation worth naming honestly:** this contract's "later half of March" trick (used
above purely to demonstrate the leakage trap inside one partition) is not how the real label
will be built in modeling weeks — a genuine forward label needs a real decision date with a full
prior-30/next-30 window on either side, which may cross month partitions and needs its own grain
and availability check before I trust it.

In [28]:
limitation_check = con.sql(f"""
    SELECT
        c.access_profile,
        COUNT(*) AS n_rows,
        AVG(CASE WHEN f.ga4_data_available IS TRUE THEN 1.0 ELSE 0 END) AS pct_ga4_available
    FROM {TABLES['fact_daily']} f
    JOIN {TABLES['dim_clients']} c USING (client_hash_id)
    WHERE f.report_date >= DATE '2026-03-01' AND f.report_date < DATE '2026-04-01'
    GROUP BY c.access_profile
    ORDER BY pct_ga4_available
""").df()

limitation_check

,access_profile,n_rows,pct_ga4_available
0,gsc_only,2128580,0.000000
1,no_search_or_analytics_access,8401,0.000000
2,source_only_missing_client_dimension,3751,0.025860
3,gsc_and_ga4,7700646,0.053745


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.